# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, following best practices for referencing dataset entities by their Croissant `@id`.

### Dataset Source
The dataset is described by a Croissant schema (JSON-LD) available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Version: {meta.version}")
print(f"Published on: {meta.date_published}")
print(f"Croissant schema URL: {croissant_url}")

## 2. Data Overview

Explore available record sets, their fields and column `@id`s. All references will use explicit Croissant `@id` fields.

In [ ]:
# List all record sets with their @id, name, and fields
print("Available record sets and fields (@id):\n")
for record_set in dataset.record_sets:
    print(f"RecordSet @id: {record_set.id} | name: {getattr(record_set, 'name', '(no name)')}")
    for field in record_set.fields:
        print(f"   Field @id:    {field.id} | name: {getattr(field, 'name', '(no name)')}")
        if getattr(field, 'column', None) is not None:
            if isinstance(field.column, list):
                for col in field.column:
                    print(f"      Column @id: {col.id} | name: {getattr(col, 'name', '(no name)')}")
            else:
                print(f"      Column @id: {field.column.id} | name: {getattr(field.column, 'name', '(no name)')}")
    print("")

# Quickly list all RecordSet @ids for further use
record_set_ids = [r.id for r in dataset.record_sets]
print("List of all RecordSet @id:")
print(record_set_ids)

## 3. Data Extraction

We extract tables from each record set using their Croissant `@id`. Here, data are loaded as pandas DataFrames and stored in a dictionary keyed by the RecordSet `@id`. All columns remain referenced by their `@id`.

Replace the example IDs with those found in the previous overview if different.

In [ ]:
# Extract record set data into pandas DataFrames
# All references are by the entity's @id

dataframes = {}
print("Loading all record sets...")

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# For demonstration, use the first available RecordSet
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nShowing columns for RecordSet @id: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No record sets loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Perform preliminary exploration — filter, transform, and summarize. For this demo, we choose a numeric field and a group field by looking at the column `@id`s.

Make sure to use the correct column `@id` as shown in the overview step for your dataset.

In [ ]:
# Choose one numeric field and one group field (if exists, replace with actual @id from the overview above)
# We'll scan for a likely numeric field if possible
df = dataframes.get(main_record_set_id)
if df is not None and not df.empty:
    print("Column @ids:", df.columns.tolist())
    # Try to auto-detect a numeric column
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        # If none auto-detected, pick the first column
        numeric_field_id = df.columns[0]
    print(f"Using numeric field @id: {numeric_field_id}")

    # Try to pick a group field if available
    possible_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'group' in col.lower() or 'anatomical' in col.lower() or df[col].dtype == object or df[col].dtype == 'category']
    group_field_id = possible_group_fields[0] if possible_group_fields else df.columns[-1]
    print(f"Using group field @id: {group_field_id}")

    # Example filter value
    threshold = 10

    filtered_df = df[df[numeric_field_id] > threshold].copy() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else df.copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (if numeric):")
    print(filtered_df.head())

    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print(f"Field {numeric_field_id} is non-numeric; skipping normalization.")

    if group_field_id in filtered_df.columns:
        # Only group by group_field_id if number of unique values is reasonable
        if filtered_df[group_field_id].nunique() < 32:
            if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
                grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
                print(grouped.head())
        else:
            print(f"Too many groups in {group_field_id} to group meaningfully.")
else:
    print("No loaded DataFrame to analyze.")

## 5. Visualization
Display basic plots using pandas and matplotlib. Visualize the chosen numeric and group fields if suitable.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if df is not None and not df.empty:
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        # Histogram of the numeric field
        plt.figure(figsize=(6,3))
        df[numeric_field_id].hist(bins=15)
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.title(f"Distribution of {numeric_field_id}")
        plt.show()
    if group_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        # Boxplot grouped by group field
        plt.figure(figsize=(7,4))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load the FAIR² dataset using `mlcroissant`, enumerate and reference record sets, fields, and columns by their Croissant `@id`, and perform entry-level exploration and processing. For further analysis, always operate using the unique `@id` references for full interoperability and programmatic access.